# fct_flight_kpis

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_gold.fct_flight_kpis;

In [0]:
-- dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/gold_flight_kpis/", True)

True

In [0]:

df_flights = spark.readStream.table("workspace.airline_silver.silver_flight_events")
df_gate = spark.read.table("workspace.airline_silver.silver_gate_operations")

In [0]:
from pyspark.sql.functions import col

df_f = df_flights.alias("f")
df_g = df_gate.alias("g")

df_joined = df_f.join(
    df_g,
    on="flight_id",
    how="left"
)

In [0]:
from pyspark.sql.functions import window, avg, count, sum, when

df_kpi = df_joined.groupBy(
    window(col("f.event_time"), "60 seconds")).agg(
    count("*").alias("total_flights"),

    avg("f.delay_minutes").alias("avg_delay_minutes"),

    sum(when(col("f.delay_minutes") <= 0, 1).otherwise(0)).alias("on_time_flights"),

    sum(when(col("f.status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_flights"),

    sum(when(col("f.delay_minutes") > 15, 1).otherwise(0)).alias("flights_at_risk")
)


In [0]:
from pyspark.sql.functions import expr

df_kpi = df_kpi.withColumn(
    "on_time_rate_pct",
    expr("on_time_flights * 100.0 / total_flights")
)


In [0]:
def upsert_to_gold(batch_df, batch_id):
    batch_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("workspace.airline_gold.fct_flight_kpis")

In [0]:
query = df_kpi.writeStream \
    .foreachBatch(upsert_to_gold) \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/gold_flight_kpis/") \
    .trigger(processingTime="60 seconds") \
    .trigger(availableNow=True) \
    .start()

In [0]:
query = df_kpi.writeStream \
    .foreachBatch(upsert_to_gold) \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/gold_flight_kpis/") \
    .trigger(availableNow=True) \
    .start()

# fct_baggage_kpis

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_gold.fct_baggage_kpis;

In [0]:
-- dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/fct_baggage_kpis/", True)

True

In [0]:
from pyspark.sql.functions import *

df_baggage = spark.readStream.table("workspace.airline_silver.silver_baggage_events").alias("b")
df_flight = spark.readStream.table("workspace.airline_silver.silver_flight_events").alias("f")


df_baggage = df_baggage.withWatermark("event_time", "2 minutes")
df_flight = df_flight.withWatermark("event_time", "2 minutes")


df_joined = df_baggage.join(df_flight, col("b.flight_id") == col("f.flight_id"))


df_kpis = df_joined.groupBy(
    window(col("b.event_time"), "60 seconds")
).agg(
    count("*").alias("total_bags"),
    
    sum(
        when(col("b.status") == "MISHANDLED", 1).otherwise(0)
    ).alias("mishandled_bags"),
    
    sum(
        when(
            (col("b.status") == "IN_TRANSIT") &
            (col("f.status") == "COMPLETED"),
            1
        ).otherwise(0)
    ).alias("bags_in_transit"),
   
    sum(
        when(
            ~col("b.status").isin("LOADED", "DELIVERED"),
            1
        ).otherwise(0)
    ).alias("backlog_bags")
)


df_final = df_kpis.withColumn(
    "mishandling_rate_pct",
    (col("mishandled_bags") * 100.0 / col("total_bags"))
)


df_final = df_final.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    "mishandling_rate_pct",
    "bags_in_transit",
    "backlog_bags"
)


query = (
    df_final.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/fct_baggage_kpis/")
    .trigger(availableNow=True)
    .table("workspace.airline_gold.fct_baggage_kpis")
)

# fct_gate_kpis

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_gold.fct_gate_kpis;

In [0]:
-- dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/fct_gate_kpis/", True)

False

In [0]:
from pyspark.sql.functions import *

df_gate = spark.readStream.table("workspace.airline_silver.silver_gate_operations").alias("g")
df_flight = spark.readStream.table("workspace.airline_silver.silver_flight_events").alias("f")


df_gate = df_gate.withWatermark("event_time", "2 minutes")
df_flight = df_flight.withWatermark("event_time", "2 minutes")


df_joined = df_gate.join(
    df_flight,
    col("g.flight_id") == col("f.flight_id")
)


df_joined = df_joined.withColumn(
    "is_upcoming",
    col("f.actual_dep").between(
        current_timestamp(),
        current_timestamp() + expr("INTERVAL 30 MINUTES")
    )
)


df_kpis = df_joined.groupBy(
    window(col("g.event_time"), "60 seconds")
).agg(

   
    count("*").alias("total_flights"),

  
    sum(
        when(col("g.event_type") == "CONFLICT", 1).otherwise(0)
    ).alias("gate_conflicts"),

   
    sum(
        when(col("g.event_type") == "GATE_CHANGE", 1).otherwise(0)
    ).alias("gate_changes"),

    
    sum(
        when(col("is_upcoming"), 1).otherwise(0)
    ).alias("upcoming_flights"),

    
    sum(
        when(
            (col("is_upcoming")) &
            (col("g.event_type") == "CREW_READY"),
            1
        ).otherwise(0)
    ).alias("ready_flights")
)



df_final = df_kpis \
    .withColumn("active_gate_conflict_count", col("gate_conflicts")) \
    .withColumn(
        "gate_change_rate_per_hour",
        try_divide(col("gate_changes") * 60.0, col("total_flights"))
    ) \
    .withColumn(
        "ground_crew_readiness_pct",
        try_divide(col("ready_flights") * 100.0, col("upcoming_flights"))
    )


df_final = df_final.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    "active_gate_conflict_count",
    "gate_change_rate_per_hour",
    "ground_crew_readiness_pct"
)


query = (
    df_final.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/airline_bronze/volumes/checkpoints/fct_gate_kpis/"
    )
    .trigger(availableNow=True)
    .table("workspace.airline_gold.fct_gate_kpis")
)

# dim_flight_status

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_gold.dim_flight_status;

In [0]:

-- dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/dim_flight_status/", True)

False

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


df_flight = spark.readStream.table(
    "workspace.airline_silver.silver_flight_events"
)


df_flight = df_flight.withWatermark("event_time", "5 minutes")

def upsert_to_dim_table(microBatchDF, batchId):

    if microBatchDF.isEmpty():
        return

    
    window_spec = Window.partitionBy("flight_id").orderBy(col("event_time").desc())

    df_dedup = microBatchDF.withColumn(
        "rn", row_number().over(window_spec)
    ).filter(col("rn") == 1).drop("rn")

    df_dedup.createOrReplaceTempView("updates")

    microBatchDF.sparkSession.sql("""
        MERGE INTO workspace.airline_gold.dim_flight_status AS target
        USING updates AS source
        ON target.flight_id = source.flight_id

        WHEN MATCHED AND source.event_time > target.event_time THEN
          UPDATE SET *

        WHEN NOT MATCHED THEN
          INSERT *
    """)



query = (
    df_flight.writeStream
    .foreachBatch(upsert_to_dim_table)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/airline_bronze/volumes/checkpoints/dim_flight_status/"
    )
    .trigger(availableNow=True)   
    .start()
)